### TEST 02E · Análogos sobre el PVAR h1b (solo diagnóstico)
Pregunta: si cada día simulado por el PVAR se sustituye por un día **real** con una situación parecida, ¿vuelven los picos de f? Si sí, el sesgo es que el PVAR genera combinaciones de PCs que no son atmósferas reales.

- Bloques de 1 día (4 pasos de 6 h, desde las 00 h). Distancia euclídea sobre las 30 PCs de presión de los 4 pasos (las PCs en bruto pesan como el campo).
- Candidatos: días reales 1941–2025 del mismo mes. Se elige al azar entre los `K` más cercanos.
- Del día análogo se pegan sus 30 PCs de presión y sus 10 PCs del gradiente reconstruido, y se pasa por el XGB `actual` del 04D.

**Ojo:** el vecino más cercano tiende a ser menos extremo que el día simulado (en la cola hay pocos candidatos). Si aun así suben los picos, el resultado es robusto.

In [1]:
import os, glob, json

import pandas as pd
import numpy as np
from scipy.signal import find_peaks
from xgboost import XGBRegressor
from sklearn.neighbors import NearestNeighbors

#### Load Data

In [2]:
pcs_msl = pd.read_csv('outputs/pca/pcs_msl_30_6h_wmed.csv', index_col=0, parse_dates=True)
pcs_grec = pd.read_csv('outputs/grad_rec/pcs_grad_rec_6h_wmed.csv', index_col=0, parse_dates=True)
cols_msl = list(pcs_msl.columns)
hist = pcs_msl.join(pcs_grec).loc['1941':'2025']

xgb_dir = 'outputs/xgb_p102_wmed_actual'
xgb = XGBRegressor()
xgb.load_model(f'{xgb_dir}/xgb_final.json')
meta = json.load(open(f'{xgb_dir}/xgb_features.json'))
lags = meta['lags_h']

SIM = 'h1b'
N_SIMS = 20
K = 5                     # se elige al azar entre los K análogos más cercanos
B = 4                     # pasos de 6 h por bloque (1 día)

#### Bloques diarios

In [3]:
def a_bloques(d):
    d = d.loc[d.index[d.index.hour == 0][0]:]
    n = len(d) // B * B
    d = d.iloc[:n]
    return d, d.values.reshape(n // B, B, d.shape[1]), d.index[::B].month

hist_b, H, mes_h = a_bloques(hist)
i_msl = [hist.columns.get_loc(c) for c in cols_msl]
H_match = H[:, :, i_msl].reshape(len(H), -1)
print(H.shape)

#### Referencia: distancia de cada día real a su día real más cercano (sin él mismo)
d_ref = np.concatenate([NearestNeighbors(n_neighbors=2).fit(H_match[mes_h == m]).kneighbors(H_match[mes_h == m])[0][:, 1]
                        for m in range(1, 13)])
print(f'distancia típica día real → análogo real: {np.median(d_ref):.1f}')

(31046, 4, 40)
distancia típica día real → análogo real: 12.0


In [4]:
def analogos(sim, seed):
    rng = np.random.default_rng(seed)
    sim_b, S, mes_s = a_bloques(sim)
    S_match = S[:, :, :len(cols_msl)].reshape(len(S), -1)
    out = np.empty((len(S), B, H.shape[2]))
    dist = np.empty(len(S))
    usados = np.empty(len(S), dtype=int)
    for m in range(1, 13):
        pool = np.where(mes_h == m)[0]
        q = np.where(mes_s == m)[0]
        d, ind = NearestNeighbors(n_neighbors=K).fit(H_match[pool]).kneighbors(S_match[q])
        elegido = ind[np.arange(len(q)), rng.integers(K, size=len(q))]
        out[q] = H[pool[elegido]]
        dist[q] = d[np.arange(len(q)), 0]
        usados[q] = pool[elegido]
    df = pd.DataFrame(out.reshape(-1, H.shape[2]), index=sim_b.index, columns=hist.columns)
    return df, dist, usados

#### f: histórico, PVAR h1b y h1b + análogos

In [5]:
def f_desde(d):
    Xs = pd.concat({f'l{l}': d.shift(freq=pd.Timedelta(hours=l)) for l in lags}, axis=1).reindex(d.index)
    Xs.columns = [f'{c}_l{l[1:]}' for l, c in Xs.columns]
    Xs = Xs.dropna()[meta['columns']]
    return pd.Series(xgb.predict(Xs), index=Xs.index)

f_hist = f_desde(hist)
umbral = f_hist.quantile(0.99)

f_pvar, f_anal, dists, frac_distintos = [], [], [], []
for i, fm in enumerate(sorted(glob.glob(f'outputs/var_6h_wmed/simulaciones_{SIM}/sim_*.parquet'))[:N_SIMS]):
    fg = os.path.join(f'outputs/grad_rec/simulaciones_{SIM}_wmed', os.path.basename(fm).replace('sim_', 'grec_'))
    sim = pd.read_parquet(fm).astype('float64')[cols_msl].join(pd.read_parquet(fg).astype('float64')).loc['1941':'2025']
    f_pvar.append(f_desde(sim))
    an, dist, usados = analogos(sim, seed=i)
    f_anal.append(f_desde(an))
    dists.append(np.median(dist))
    frac_distintos.append(len(np.unique(usados)) / len(H))

print(f'distancia mediana sim → análogo: {np.median(dists):.1f}  (real → real: {np.median(d_ref):.1f})')
print(f'fracción de días reales distintos usados por simulación: {np.median(frac_distintos):.2f}')

distancia mediana sim → análogo: 14.2  (real → real: 12.0)
fracción de días reales distintos usados por simulación: 0.52


In [6]:
def resumen(f):
    pk = find_peaks(f.values, height=umbral, distance=12)[0]
    return {'q90': f.quantile(0.90), 'q99': f.quantile(0.99), 'q99.9': f.quantile(0.999),
            'máx anual medio': f.groupby(f.index.year).max().mean(),
            'picos f > q99 hist / década': len(pk) * 10 / f.index.year.nunique()}

h = pd.Series(resumen(f_hist))
pd.DataFrame({'histórico': h,
              f'{SIM} / hist': pd.DataFrame([resumen(f) for f in f_pvar]).median() / h,
              f'{SIM} + análogos / hist': pd.DataFrame([resumen(f) for f in f_anal]).median() / h}).round(3)

,histórico,h1b / hist,h1b + análogos / hist
q90,1.050,0.906,0.730
q99,1.915,0.967,0.802
q99.9,2.496,1.007,0.848
máx anual medio,2.495,0.995,0.868
picos f > q99 hist / década,49.059,0.794,0.344


In [7]:
est = lambda m: np.select([np.isin(m, [12, 1, 2]), np.isin(m, [3, 4, 5]), np.isin(m, [6, 7, 8])], ['invierno', 'primavera', 'verano'], 'otoño')

def picos_est(f):
    pk = f.index[find_peaks(f.values, height=umbral, distance=12)[0]]
    return pd.Series(est(pk.month)).value_counts() * 10 / f.index.year.nunique()

pd.DataFrame({'histórico': picos_est(f_hist),
              SIM: pd.concat([picos_est(f) for f in f_pvar], axis=1).fillna(0).median(axis=1),
              f'{SIM} + análogos': pd.concat([picos_est(f) for f in f_anal], axis=1).fillna(0).median(axis=1)}
            ).loc[['invierno', 'primavera', 'verano', 'otoño']].round(1)

,histórico,h1b,h1b + análogos
invierno,21.2,17.3,9.4
primavera,12.9,7.4,4.1
verano,4.2,5.7,0.6
otoño,10.7,8.1,2.4


In [8]:
#### 1. Episodios sobre el q99: ¿menos episodios pero igual tiempo por encima?
def episodios(f, u=umbral):
    sobre = f > u
    g = (sobre != sobre.shift()).cumsum()[sobre]
    dur = sobre[sobre].groupby(g).size() * 6                      # horas
    n_dec = f.index.year.nunique() / 10
    return {'% tiempo > q99': sobre.mean() * 100,
            'episodios / década': len(dur) / n_dec,
            'duración media [h]': dur.mean(),
            'duración p90 [h]': dur.quantile(0.9),
            'picos find_peaks / década': len(find_peaks(f.values, height=u, distance=12)[0]) / n_dec}

display(pd.DataFrame({'histórico': episodios(f_hist),
                      'h1b': pd.DataFrame([episodios(f) for f in f_pvar]).median()}).round(2))

#### 2. Forma media de f alrededor de los picos (−72 h a +72 h): ¿suben y bajan igual de rápido?
def compuesto(f, u=umbral, w=12):
    pk = find_peaks(f.values, height=u, distance=12)[0]
    pk = pk[(pk >= w) & (pk < len(f) - w)]
    return np.mean([f.values[p - w:p + w + 1] for p in pk], axis=0)

desfase_h = np.arange(-12, 13) * 6
comp = pd.DataFrame({'histórico': compuesto(f_hist),
                     'h1b': np.mean([compuesto(f) for f in f_pvar], axis=0)}, index=desfase_h)
display(comp.loc[::2].round(2).T)

#### 3. Persistencia de f (6 h a 72 h)
acf = lambda f, l: f.autocorr(l)
display(pd.DataFrame({'histórico': [acf(f_hist, l) for l in range(1, 13)],
                      'h1b': [np.median([acf(f, l) for f in f_pvar]) for l in range(1, 13)]},
                     index=[f'{6 * l} h' for l in range(1, 13)]).round(3).T)

,histórico,h1b
% tiempo > q99,1.00,0.85
episodios / década,52.00,41.82
duración media [h],16.86,17.73
duración p90 [h],30.00,36.00
picos find_peaks / década,49.06,38.94


,-72,-60,-48,-36,-24,-12,0,12,24,36,48,60,72
histórico,0.15,0.18,0.24,0.38,0.78,1.54,2.19,1.69,1.05,0.55,0.31,0.18,0.12
h1b,0.23,0.23,0.28,0.38,0.74,1.54,2.24,1.67,0.93,0.46,0.27,0.20,0.20


,6 h,12 h,18 h,24 h,30 h,36 h,42 h,48 h,54 h,60 h,66 h,72 h
histórico,0.953,0.856,0.741,0.624,0.510,0.414,0.341,0.285,0.238,0.201,0.177,0.158
h1b,0.940,0.821,0.686,0.555,0.436,0.343,0.276,0.229,0.191,0.165,0.150,0.140


In [9]:
#### Surrogado de fase multivariante (IAAFT simple): mismo 2º orden y mismas marginales que el histórico, nada más
from utils.tv_var.harmonic_standardizer import HarmonicStandardizer

hs_all = HarmonicStandardizer(n_harmonics=3)
hs_all.fit(hist)
z = hs_all.transform(hist)

def surrogado(z, seed):
    rng = np.random.default_rng(seed)
    F = np.fft.rfft(z.values - z.values.mean(0), axis=0)
    fase = np.exp(1j * rng.uniform(0, 2 * np.pi, F.shape[0]))[:, None]   # misma fase en todas → conserva espectros cruzados
    fase[0] = 1
    s = np.fft.irfft(F * fase, n=len(z), axis=0) + z.values.mean(0)
    orden = np.argsort(np.argsort(s, axis=0), axis=0)                     # devolver la marginal de cada PC
    s = np.take_along_axis(np.sort(z.values, axis=0), orden, axis=0)
    return hs_all.inverse_transform(pd.DataFrame(s, index=z.index, columns=z.columns))

f_sur = [f_desde(surrogado(z, i)) for i in range(10)]

display(pd.DataFrame({'h1b / hist': pd.DataFrame([resumen(f) for f in f_pvar]).median() / h,
                      'surrogado / hist': pd.DataFrame([resumen(f) for f in f_sur]).median() / h}).round(3))
display(pd.DataFrame({'histórico': episodios(f_hist),
                      'h1b': pd.DataFrame([episodios(f) for f in f_pvar]).median(),
                      'surrogado': pd.DataFrame([episodios(f) for f in f_sur]).median()}).round(2))
pd.DataFrame({'histórico': picos_est(f_hist),
              'h1b': pd.concat([picos_est(f) for f in f_pvar], axis=1).fillna(0).median(axis=1),
              'surrogado': pd.concat([picos_est(f) for f in f_sur], axis=1).fillna(0).median(axis=1)}
            ).loc[['invierno', 'primavera', 'verano', 'otoño']].round(1)

,h1b / hist,surrogado / hist
q90,0.906,0.842
q99,0.967,0.792
q99.9,1.007,0.788
máx anual medio,0.995,0.797
picos f > q99 hist / década,0.794,0.187


,histórico,h1b,surrogado
% tiempo > q99,1.00,0.85,0.13
episodios / década,52.00,41.82,9.47
duración media [h],16.86,17.73,11.95
duración p90 [h],30.00,36.00,24.00
picos find_peaks / década,49.06,38.94,9.18


,histórico,h1b,surrogado
invierno,21.2,17.3,4.5
primavera,12.9,7.4,2.6
verano,4.2,5.7,0.4
otoño,10.7,8.1,1.5
